# 05 · Fase 1: efecto propio y efectos dentro de la categoría

Estima, para una categoría objetivo $j$, el modelo PPML de la **Fase 1** de `metodologia.tex`: efecto de la promoción del propio SKU (día actual y dos bloques de rezagos) y de la promoción de los otros SKUs de su clase y del resto de su categoría.

| | |
|---|---|
| **Entrada** | `panel_t44_sku_dia.parquet`, `exposiciones_t44_sku_dia.parquet`, `calendario_t44.csv` (en `../data/procesados/`) |
| **Salida** | `resultados/tablas/fase1_<CATEGORIA>.csv` y `resultados/tablas/fase1_consolidada.csv` |
| **Especificación** | PPML, variable dependiente `unit_sales_ppml`; efectos fijos SKU, día de la semana y mes; errores agrupados por SKU y semana |

**Cómo usarlo.** Empiecen con `categorias_objetivo = ["BEVERAGES"]`, revisen los controles de la sección 3 y recién entonces agreguen las demás categorías a la lista. No se incluye `transactions` ni el petróleo.

## Registro de cambios respecto a la versión original (revisión del 2026-10-22)

Los cambios están marcados en el código con `# [CAMBIO n]`.

| # | Dónde | Qué se hizo | Por qué |
|---|---|---|---|
| 1 | `estimar_fase1`, tras `efecto_neto_propio` | Se calcula `g`, el efecto acumulado de una promoción de un día, con error estándar (método delta) y p-valor. | `_l1_3` y `_l4_6` son promedios de 3 días. `efecto_neto_propio` es el efecto de estar en promoción 7 días seguidos, no de una promoción de un día. En BEVERAGES, `g` = +10,6 %. |
| 2 | `estimar_fase1`, tras `efecto_pct` e IC | Se agrega la fila `efecto_acum_1dia`, ya en porcentaje (coef = NaN). | `g` ya es un porcentaje; agregarla antes le aplicaría `exp(coef)-1` dos veces. |
| 3 | `estimar_fase1`, variable `interes` | Se incluye `efecto_acum_1dia` en lo reportado. | Sin esto el filtro final la elimina. |
| 4 | Parámetros y ciclo de estimación | `dias_maximos = 532` y `assert ==` (con `<=` solo para categorías con programa). | 590 − 46 − 12 por los rezagos `_l4_6` (6 al inicio y 6 tras el terremoto). Con `<=` no se detectaba un día faltante. |
| 5 | `estimar_fase1`, columnas de muestra | `n_obs = modelo._N` y nueva columna `n_obs_descartadas`. | Se reportaba `len(base)`, que incluye filas que pyfixest descarta (singletons). |
| 6 | `estimar_fase1`, antes del `return` | Aviso si algún error estándar no es finito. | Con errores agrupados por SKU y semana la varianza puede salir negativa y dar NaN sin que nadie lo note. No detiene el ciclo. |
| 7 | `estimar_fase1`, p-valores derivados | Se usa la t con los grados de libertad del modelo (normal si no están disponibles). | Las filas derivadas usaban la normal y el resto de la tabla la t de pyfixest. |
| 8 | `construir_base`, paso 6 | `base = base.copy()`. | Evita el `SettingWithCopyWarning`. |
| 9 | Parámetros y sección 4 | Lista `categorias_prioridad`; la consolidada solo lee `fase1_<CATEGORIA>.csv` exactos, revisa duplicados y avisa de corridas sin `efecto_acum_1dia`. | Leer todo lo que empiece con `fase1_` mezclaba copias (p. ej. `fase1_BEVERAGES1.csv`) y corridas viejas. |
| 10 | Sección 4, comparación | Se compara la razón del modelo (1 + efecto/100) con la razón de medias. | Antes se comparaba un porcentaje con una razón. |
| 11 | `construir_base`, antes del `dropna` | Se imprime cuántas filas tienen nulos y en qué variables. | Confirmar de dónde sale el ~3 % de filas perdidas en BEVERAGES y el ~15 % en LIQUOR. Solo informativo. |

`efecto_neto_propio` se mantiene para comparar; en la tesis debe rotularse como "promoción sostenida de 7 días".

**Pendientes fuera de este notebook:** revisar la salida del diagnóstico de nulos (cambio 11) y ajustar el texto de la tesis (532 días, `mes` = año-mes, lectura del efecto neto, limitaciones). Los outputs guardados son de la versión anterior; hay que volver a ejecutar.

In [1]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima el modelo PPML de la fase 1 (efecto propio y dentro de la categoría) por categoría.
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import re

import numpy as np
import pandas as pd
import pyfixest as pf

## 1. Parámetros

In [2]:
# Categorías a estimar (empezar solo con BEVERAGES; luego agregar las demás de la lista de prioridad)
# ------------------------------------------------------------------------------------------------------------
categorias_objetivo = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
# Lista completa por prioridad (INSTRUCCIONES.md, sección 6):
# ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER",
#  "MEATS", "POULTRY", "DELI", "PRODUCE"]

# Categorías con programa semanal fijo (D-26): se excluyen sus días de programa
categorias_con_programa = ["MEATS", "POULTRY", "SEAFOOD", "EGGS", "DELI", "PRODUCE"]

# Bloques de promoción (D-14): día actual y dos bloques de rezagos
sufijos = ["", "_l1_3", "_l4_6"]

# Indicadores de calendario (h_t); 'terremoto' queda fuera porque su ventana se excluye de la muestra
indicadores_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

# Efectos fijos y agrupamiento de los errores estándar (D-15, D-16)
efectos_fijos = ["item_nbr", "dow", "mes"]
agrupar_por = ["item_nbr", "semana"]

# [CAMBIO 4] Días de estimación esperados: 590 - 46 (ventana del terremoto) - 12 por los rezagos _l4_6
# (6 días al inicio del período y 6 justo después del terremoto, cuyos bloques quedan nulos). Antes decía 544.
dias_maximos = 532

# [CAMBIO 9] Lista completa de categorías válidas; la consolidación solo lee los archivos de estas (ver sección 4)
categorias_prioridad = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER",
                        "MEATS", "POULTRY", "DELI", "PRODUCE"]

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
os.makedirs(ruta_tablas, exist_ok=True)
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_exposiciones = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")

## 2. Base de estimación de una categoría

Sigue los pasos de la última celda de `03_exposiciones` (sin las exposiciones a otras categorías, que entran en la fase 2). Cada unión verifica que no se dupliquen ni se pierdan filas, y cada exclusión se cuenta.

In [3]:
def construir_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de la fase 1 para una categoría objetivo.

    :param categoria:   el nombre de la categoría objetivo (p. ej., 'BEVERAGES').
    :return:            una tupla (base, exclusiones): la base SKU-día lista para estimar y un diccionario con el
                        número de filas en cada paso.
    """

    exclusiones = {}

    # 1. Filas del panel de la categoría
    panel = pd.read_parquet(
        archivo_panel,
        columns=["date", "item_nbr", "family", "unit_sales_ppml", "onpromotion"],
        filters=[("family", "==", categoria)],
    )
    panel["family"] = panel["family"].astype(str)
    assert len(panel) > 0, f"no hay filas para la categoría {categoria}"
    assert not panel.duplicated(["item_nbr", "date"]).any(), "el panel tiene más de una fila por SKU-día"
    exclusiones["panel de la categoría"] = len(panel)

    # 2. Unir las exposiciones SKU-día (uno a uno por SKU y fecha)
    columnas_expo = ["item_nbr", "date", "dia_programa", "promo_clase_otros", "promo_cat_otras_clases"] + [
        f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos[1:]
    ]
    expo = pd.read_parquet(archivo_exposiciones, columns=columnas_expo)
    n0 = len(panel)
    base = panel.merge(expo, on=["item_nbr", "date"], how="left", validate="one_to_one")
    assert len(base) == n0, "la unión con exposiciones cambió el número de filas"
    assert base["dia_programa"].notna().all(), "hay SKU-días del panel sin exposiciones"

    # 3. Unir el calendario (muchos a uno por fecha); las fechas se leen con parse_dates
    calendario = pd.read_csv(archivo_calendario, parse_dates=["date"])
    base = base.merge(
        calendario[["date", "dow", "mes", "semana", "ventana_terremoto"] + indicadores_calendario],
        on="date", how="left", validate="many_to_one",
    )
    assert len(base) == n0, "la unión con el calendario cambió el número de filas"
    assert base["dow"].notna().all(), "hay fechas del panel sin calendario"
    assert not base.duplicated(["item_nbr", "date"]).any(), "la base tiene más de una fila por SKU-día"

    # 4. Excluir la ventana del terremoto
    base = base.loc[~base["ventana_terremoto"]]
    exclusiones["tras excluir ventana del terremoto"] = len(base)

    # 5. Si la categoría tiene programa semanal, excluir sus días de programa
    if categoria in categorias_con_programa:
        base = base.loc[~base["dia_programa"]]
    exclusiones["tras excluir días de programa"] = len(base)

    # 6. Excluir filas con nulos en las variables del modelo
    # [CAMBIO 8] .copy() para que la asignación siguiente no dispare el SettingWithCopyWarning
    base = base.copy()
    base["onpromotion"] = base["onpromotion"].astype(float)
    variables = variables_modelo()

    # [CAMBIO 11] Diagnóstico: de qué variables vienen los nulos que se eliminan en el siguiente paso (se descarta la
    # fila si falta cualquiera). Sirve para confirmar si son los primeros días de vida de cada SKU, los bordes del
    # período o del terremoto, o las clases / categorías sin otros SKUs activos. Es solo informativo.
    columnas_nulos = variables + ["unit_sales_ppml"]
    nulos = base[columnas_nulos].isna().sum()
    print(f"  Filas con algún nulo: {base[columnas_nulos].isna().any(axis=1).sum():,}")
    if nulos.any():
        print(nulos[nulos > 0].to_string())

    base = base.dropna(subset=variables + ["unit_sales_ppml"])
    exclusiones["tras excluir nulos (base final)"] = len(base)

    assert not base.duplicated(["item_nbr", "date"]).any(), "la base final tiene más de una fila por SKU-día"
    return base.reset_index(drop=True), exclusiones


def variables_modelo() -> list:
    """
    Esta función devuelve la lista de regresores de la fase 1 (promoción propia, de la clase y del resto de la
    categoría, cada una con sus bloques de rezagos).

    :return:    la lista de nombres de columna.
    """

    return [f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos]

## 3. Estimación y tabla de resultados

PPML con `pyfixest.fepois`. Se descartan los indicadores de calendario sin variación en la muestra (p. ej., `terremoto` ya no existe, y un feriado puede caer solo en días excluidos). El efecto neto de la promoción propia es $e^{\beta_0+\beta_{\ell_{1-3}}+\beta_{\ell_{4-6}}}-1$, con su error estándar por el método delta.

**Efecto acumulado de una promoción de un día (`efecto_acum_1dia`).** Como `_l1_3` y `_l4_6` son promedios de 3 días, $e^{\beta_0+\beta_{\ell_{1-3}}+\beta_{\ell_{4-6}}}-1$ es el efecto de estar en promoción 7 días seguidos, no de una promoción de un día. El efecto de una promoción de un día, respecto de una semana normal, es $\frac{1}{7}\left[(e^{\beta_0}-1)+3(e^{\beta_{\ell_{1-3}}/3}-1)+3(e^{\beta_{\ell_{4-6}}/3}-1)\right]$, con error estándar por el método delta. Es una aproximación: supone el mismo nivel base en los 7 días.

In [4]:
def estimar_fase1(base: pd.DataFrame, categoria: str) -> pd.DataFrame:
    """
    Esta función estima el PPML de la fase 1 y devuelve la tabla de resultados.

    :param base:        la base de estimación de la categoría (salida de construir_base).
    :param categoria:   el nombre de la categoría objetivo.
    :return:            un DataFrame con una fila por coeficiente de interés (y una fila de efecto neto propio) con
                        coeficiente, error estándar, valor p, efecto en porcentaje y tamaño de la muestra.
    """

    # Conservar solo los indicadores de calendario con variación
    calendario_activo = [c for c in indicadores_calendario if base[c].nunique() > 1]
    descartados = sorted(set(indicadores_calendario) - set(calendario_activo))
    if descartados:
        print(f"  Indicadores sin variación (descartados): {descartados}")
    base = base.assign(**{c: base[c].astype(float) for c in calendario_activo})

    # Estimar
    regresores = variables_modelo() + calendario_activo
    formula = f"unit_sales_ppml ~ {' + '.join(regresores)} | {' + '.join(efectos_fijos)}"
    modelo = pf.fepois(formula, data=base, vcov={"CRV1": "+".join(agrupar_por)})

    # Reunir los coeficientes de interés
    tabla = modelo.tidy().reset_index().rename(columns={"Coefficient": "variable"})
    tabla = tabla.rename(columns={"Estimate": "coef", "Std. Error": "se", "Pr(>|t|)": "p_valor"})
    tabla = tabla[["variable", "coef", "se", "p_valor"]]

    # Efecto neto propio con el método delta: combinación lineal de tres coeficientes
    nombres = list(modelo._coefnames)
    propios = [f"onpromotion{s}" for s in sufijos]
    w = np.array([1.0 if n in propios else 0.0 for n in nombres])
    vcov = np.asarray(modelo._vcov)
    coef_neto = float(w @ modelo.coef().reindex(nombres).to_numpy())
    se_neto = float(np.sqrt(w @ vcov @ w))
    from scipy import stats
    # [CAMBIO 7] Los p-valores de las filas derivadas usan la misma distribución que los de pyfixest (t con los grados
    # de libertad del modelo, que con errores agrupados dependen del número de clusters). Antes usaban la normal y no
    # eran comparables con el resto de la tabla. Si pyfixest no expone los grados de libertad, se usa la normal.
    gl = getattr(modelo, "_df_t", None)

    def p_dos_colas(z: float) -> float:
        return float(2 * stats.t.sf(abs(z), gl)) if gl else float(2 * stats.norm.sf(abs(z)))

    p_neto = p_dos_colas(coef_neto / se_neto)
    tabla = pd.concat(
        [tabla, pd.DataFrame([{"variable": "efecto_neto_propio", "coef": coef_neto, "se": se_neto, "p_valor": p_neto}])],
        ignore_index=True,
    )

    # [CAMBIO 1] Efecto acumulado de UNA promoción de un día, sobre una semana normal de 7 días.
    # Por qué: los bloques _l1_3 y _l4_6 son PROMEDIOS de 3 días, así que una promoción de un día entra con peso 1/3
    # en cada día de su bloque. Por eso exp(b0+b1+b2)-1 (efecto_neto_propio) mide una promoción sostenida 7 días
    # seguidos, no una promoción de un día. Esta fórmula suma los efectos diarios implícitos y divide por 7.
    # El método delta sí es necesario aquí porque la fórmula no es lineal en los coeficientes.
    idx = [nombres.index(n) for n in propios]
    assert len(idx) == 3, "falta algún regresor propio (colinealidad)"
    b = modelo.coef().reindex(nombres).to_numpy()[idx]
    V = np.asarray(modelo._vcov)[np.ix_(idx, idx)]
    g = (np.exp(b[0]) - 1 + 3 * (np.exp(b[1] / 3) - 1) + 3 * (np.exp(b[2] / 3) - 1)) / 7
    grad = np.array([np.exp(b[0]), np.exp(b[1] / 3), np.exp(b[2] / 3)]) / 7
    se_g = float(np.sqrt(grad @ V @ grad))
    p_g = p_dos_colas(g / se_g)

    # Efecto en porcentaje, 100 (exp(beta) - 1), e intervalo de confianza al 95%
    tabla["efecto_pct"] = 100 * (np.exp(tabla["coef"]) - 1)
    tabla["ic95_inf_pct"] = 100 * (np.exp(tabla["coef"] - 1.96 * tabla["se"]) - 1)
    tabla["ic95_sup_pct"] = 100 * (np.exp(tabla["coef"] + 1.96 * tabla["se"]) - 1)

    # [CAMBIO 2] Agregar la fila ya en porcentaje. Va DESPUÉS del cálculo de efecto_pct/IC de arriba para que
    # no se le aplique otra vez exp(coef)-1 (g ya es un porcentaje, no un coeficiente en logaritmos).
    fila_acum = pd.DataFrame([{
        "variable": "efecto_acum_1dia", "coef": np.nan, "se": se_g, "p_valor": p_g,
        "efecto_pct": 100 * g,
        "ic95_inf_pct": 100 * (g - 1.96 * se_g),
        "ic95_sup_pct": 100 * (g + 1.96 * se_g),
    }])
    tabla = pd.concat([tabla, fila_acum], ignore_index=True)

    # Tamaño de la muestra
    tabla.insert(0, "categoria", categoria)
    # [CAMBIO 5] n_obs es la muestra que realmente usó el modelo (pyfixest descarta efectos fijos singleton); antes
    # se reportaba len(base). Se guarda también cuántas filas descartó.
    n_modelo = int(getattr(modelo, "_N", len(base)))
    tabla["n_obs"] = n_modelo
    tabla["n_obs_descartadas"] = len(base) - n_modelo
    tabla["n_skus"] = base["item_nbr"].nunique()
    tabla["n_dias"] = base["date"].nunique()

    # Solo se reportan las variables de promoción (los indicadores de calendario son controles)
    # [CAMBIO 3] Se agrega "efecto_acum_1dia" a la lista; si no, el filtro final eliminaría la fila nueva.
    interes = variables_modelo() + ["efecto_neto_propio", "efecto_acum_1dia"]
    resultado = tabla.loc[tabla["variable"].isin(interes)].reset_index(drop=True)

    # [CAMBIO 6] Aviso fuerte si algún error estándar no es finito (varianza negativa con errores agrupados por
    # SKU y semana). No detiene el ciclo para no perder las demás categorías. Si aparece, estimar esa categoría
    # con vcov={"CRV1": "semana"} como respaldo y declararlo en la tesis. El coef NaN de efecto_acum_1dia es normal.
    fallan = resultado.loc[~np.isfinite(resultado["se"]), "variable"].tolist()
    if fallan:
        print(f"  >>> ATENCIÓN {categoria}: error estándar no finito en {fallan}; no interpretar esas filas")
    return resultado

In [5]:
# Estimar cada categoría y guardar su tabla
resultados = {}
for categoria in categorias_objetivo:
    print(f"\n=== {categoria} ===")
    base, exclusiones = construir_base(categoria)

    # Controles de la sección 10.1 de INSTRUCCIONES.md
    for paso, n in exclusiones.items():
        print(f"  {paso:<40} {n:>10,}")
    dias_usados = base["date"].nunique()
    # [CAMBIO 4] Con == se detecta tanto un día de más como uno de menos. Las categorías con programa semanal pierden
    # además sus días de programa, así que para ellas el tope es <=.
    if categoria in categorias_con_programa:
        assert dias_usados <= dias_maximos, f"se usan {dias_usados} días y el máximo es {dias_maximos}"
    else:
        assert dias_usados == dias_maximos, f"se usan {dias_usados} días y se esperaban {dias_maximos}: revisar exclusiones"
    print(f"  SKUs: {base['item_nbr'].nunique():,} | días usados: {dias_usados} (esperados {dias_maximos})")

    tabla = estimar_fase1(base, categoria)
    resultados[categoria] = tabla
    nombre_archivo = re.sub(r"[^A-Za-z0-9]+", "_", categoria).strip("_")
    tabla.to_csv(os.path.join(ruta_tablas, f"fase1_{nombre_archivo}.csv"), index=False)

    # El efecto propio contemporáneo debería ser positivo; si no, revisar antes de interpretar
    propio = tabla.loc[tabla["variable"] == "onpromotion"].iloc[0]
    print(tabla[["variable", "coef", "se", "p_valor", "efecto_pct"]].round(4).to_string(index=False))
    if propio["coef"] <= 0:
        print("  >>> ATENCIÓN: el efecto propio del día es <= 0; revisar la construcción de la variable antes de interpretar")
    if (tabla["coef"].abs() > 5).any():
        print("  >>> ATENCIÓN: hay coeficientes con |valor| > 5 (posible colinealidad o separación)")


=== BEVERAGES ===
  Filas con algún nulo: 7,829
onpromotion_l1_3               2575
onpromotion_l4_6               5173
promo_clase_otros              1731
promo_clase_otros_l1_3         4782
promo_clase_otros_l4_6         7829
promo_cat_otras_clases_l1_3    3072
promo_cat_otras_clases_l4_6    6140
  panel de la categoría                       287,407
  tras excluir ventana del terremoto          266,744
  tras excluir días de programa               266,744
  tras excluir nulos (base final)             258,915
  SKUs: 576 | días usados: 532 (esperados 532)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.4688 0.0580   0.0000     59.8155
           onpromotion_l1_3  0.2384 0.0618   0.0001     26.9232
           onpromotion_l4_6 -0.1045 0.0627   0.0955     -9.9227
          promo_clase_otros -0.3055 0.1143   0.0075    -26.3266
     promo_clase_otros_l1_3 -0.0197 0.1204   0.8701     -1.9488
     promo_clase_otros_l4_6  0.0786 0.1014   0.4384      8.1747
     promo_cat_otras_clases -0.6335 0.3613   0.0796    -46.9260
promo_cat_otras_clases_l1_3  0.2720 0.4135   0.5106     31.2614
promo_cat_otras_clases_l4_6  0.4063 0.2976   0.1722     50.1306
         efecto_neto_propio  0.6028 0.0781   0.0000     82.7154
           efecto_acum_1dia     NaN 0.0155   0.0000     10.6228

=== GROCERY I ===
  Filas con algún nulo: 17,685
onpromotion_l1_3                5759
onpromotion_l4_6               11630
promo_clase_otros               4749
promo_clase_otros_l1_3         11221
promo_clase_otros_l4_6         176

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.4727 0.0507   0.0000     60.4291
           onpromotion_l1_3  0.2597 0.0428   0.0000     29.6526
           onpromotion_l4_6 -0.0109 0.0431   0.8005     -1.0837
          promo_clase_otros -0.2255 0.0717   0.0017    -20.1904
     promo_clase_otros_l1_3 -0.3140 0.0735   0.0000    -26.9496
     promo_clase_otros_l4_6  0.1692 0.0750   0.0240     18.4306
     promo_cat_otras_clases -0.7388 0.3946   0.0612    -52.2303
promo_cat_otras_clases_l1_3 -0.3999 0.6613   0.5453    -32.9644
promo_cat_otras_clases_l4_6  1.2656 0.6077   0.0373    254.5307
         efecto_neto_propio  0.7215 0.0446   0.0000    105.7464
           efecto_acum_1dia     NaN 0.0090   0.0000     12.3525

=== CLEANING ===
  Filas con algún nulo: 5,896
onpromotion_l1_3               2029
onpromotion_l4_6               4105
promo_clase_otros              1366
promo_clase_otros_l1_3         3630
promo_clase_otros_l4_6         5895
prom

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.6363 0.0742   0.0000     88.9383
           onpromotion_l1_3  0.2331 0.1028   0.0234     26.2466
           onpromotion_l4_6 -0.0243 0.0692   0.7260     -2.3976
          promo_clase_otros -0.3105 0.1354   0.0218    -26.6889
     promo_clase_otros_l1_3 -0.4868 0.1979   0.0139    -38.5432
     promo_clase_otros_l4_6  0.1126 0.1488   0.4492     11.9200
     promo_cat_otras_clases -0.4895 0.3676   0.1829    -38.7086
promo_cat_otras_clases_l1_3  0.3007 0.4111   0.4644     35.0867
promo_cat_otras_clases_l4_6  0.7745 0.4284   0.0706    116.9493
         efecto_neto_propio  0.8450 0.0678   0.0000    132.8094
           efecto_acum_1dia     NaN 0.0162   0.0000     15.8225

=== DAIRY ===
  Filas con algún nulo: 3,671
onpromotion_l1_3               1136
onpromotion_l4_6               2287
promo_clase_otros              1138
promo_clase_otros_l1_3         2405
promo_clase_otros_l4_6         3671
promo_c

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.3747 0.0584   0.0000     45.4517
           onpromotion_l1_3  0.1445 0.0661   0.0287     15.5460
           onpromotion_l4_6  0.0985 0.0749   0.1885     10.3544
          promo_clase_otros -0.0753 0.0893   0.3993     -7.2543
     promo_clase_otros_l1_3 -0.0494 0.0844   0.5582     -4.8207
     promo_clase_otros_l4_6 -0.1399 0.0991   0.1579    -13.0564
     promo_cat_otras_clases -0.6642 0.2305   0.0040    -48.5324
promo_cat_otras_clases_l1_3  0.4088 0.3625   0.2595     50.5014
promo_cat_otras_clases_l4_6 -0.1755 0.2529   0.4877    -16.0945
         efecto_neto_propio  0.6177 0.0991   0.0000     85.4658
           efecto_acum_1dia     NaN 0.0173   0.0000     10.0388

=== LIQUOR,WINE,BEER ===
  Filas con algún nulo: 3,682
onpromotion_l1_3                215
onpromotion_l4_6                453
promo_clase_otros              3164
promo_clase_otros_l1_3         3421
promo_clase_otros_l4_6         3

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


## 4. Comparación con los descriptivos y tabla consolidada

La dirección del efecto propio debe parecerse a la razón de medias de la tabla 4 de `04_descriptivos` (la magnitud puede diferir, porque el modelo controla SKU y calendario).

In [6]:
# Comparar con la tabla 4 de los descriptivos (razón de medias con / sin promoción)
archivo_t4 = os.path.join(ruta_tablas, "tabla_04_ventas_con_sin_promocion.csv")
if os.path.exists(archivo_t4):
    t4 = pd.read_csv(archivo_t4, index_col="family")
    for categoria, tabla in resultados.items():
        if categoria in t4.index:
            efecto = tabla.loc[tabla["variable"] == "onpromotion", "efecto_pct"].iloc[0]
            # [CAMBIO 10] Se compara en la misma escala: razón de ventas del modelo (1 + efecto/100) vs razón de medias
            razon_modelo = 1 + efecto / 100
            print(f"{categoria}: modelo {razon_modelo:.2f}x ({efecto:+.1f}%) | descriptivo (razón de medias) {t4.loc[categoria, 'razon_medias']:.2f}")
else:
    print("No se encontró la tabla 4; ejecutar 04_descriptivos para comparar.")

# Consolidar todas las tablas por categoría que existan en la carpeta (así no se pierde lo estimado antes)
# [CAMBIO 9] Solo se leen los archivos con el nombre exacto de cada categoría válida (fase1_<CATEGORIA>.csv). Antes se
# leía todo lo que empezara con "fase1_", y copias como fase1_BEVERAGES1.csv duplicaban filas en la consolidada.
validos = {f"fase1_{re.sub(r'[^A-Za-z0-9]+', '_', c).strip('_')}.csv" for c in categorias_prioridad}
archivos = sorted(f for f in os.listdir(ruta_tablas) if f in validos)
consolidada = pd.concat([pd.read_csv(os.path.join(ruta_tablas, f)) for f in archivos], ignore_index=True)
assert not consolidada.duplicated(["categoria", "variable"]).any(), "la consolidada tiene filas repetidas"
con_acum = set(consolidada.loc[consolidada["variable"] == "efecto_acum_1dia", "categoria"])
viejas = sorted(set(consolidada["categoria"]) - con_acum)
if viejas:
    print(f"ATENCIÓN: estas categorías vienen de una corrida anterior (sin efecto_acum_1dia): {viejas}; volver a estimarlas")
consolidada.to_csv(os.path.join(ruta_tablas, "fase1_consolidada.csv"), index=False)
print(f"\nConsolidada: {consolidada['categoria'].nunique()} categorías -> {os.path.join(ruta_tablas, 'fase1_consolidada.csv')}")

print(f"\n>>>> Proceso finalizado <<<<")

BEVERAGES: modelo 1.60x (+59.8%) | descriptivo (razón de medias) 2.35
GROCERY I: modelo 1.60x (+60.4%) | descriptivo (razón de medias) 2.36
CLEANING: modelo 1.89x (+88.9%) | descriptivo (razón de medias) 2.36
DAIRY: modelo 1.41x (+41.2%) | descriptivo (razón de medias) 1.28
BREAD/BAKERY: modelo 1.45x (+45.5%) | descriptivo (razón de medias) 1.91
LIQUOR,WINE,BEER: modelo 1.75x (+75.0%) | descriptivo (razón de medias) 2.20

Consolidada: 6 categorías -> resultados\tablas\fase1_consolidada.csv

>>>> Proceso finalizado <<<<
